# 04 — Pipeline & Refinement Loop

This notebook demonstrates two orchestration primitives from orqest:

1. **Pipeline** — chains multiple steps (agents and functions) sequentially, feeding each step's output as the next step's input
2. **RefinementLoop** — iterates a step with evaluation feedback until quality passes or limits are reached

**Use case:** We build a Research Brief Generator that analyzes a topic, formulates research questions, and writes a structured brief. Then we use a RefinementLoop to iteratively improve writing quality.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4o   # or anthropic:claude-sonnet-4-20250514, google:gemini-2.0-flash, etc.
  ```

## 1. Setup

Load configuration from `.env` — orqest never loads environment variables at import time.

In [1]:
from orqest import load_config

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

Model:    openai:gpt-4.1
API key:  sk-proj-...


## 2. Define the agents

Our pipeline has two agents and one pure function:

1. **TopicAnalyzer** — takes a research topic string and identifies 3 key subtopics
2. **format_questions()** — a pure async function that converts subtopics into research questions
3. **BriefWriter** — takes the research questions and writes a structured brief

Each agent defines its own Pydantic output type for structured responses.

In [2]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


# --- Output types ---

class TopicAnalysis(BaseModel):
    """Output of the TopicAnalyzer agent."""
    topic: str = Field(description="The original research topic")
    subtopics: list[str] = Field(description="Exactly 3 key subtopics to investigate")
    rationale: str = Field(description="Brief explanation of why these subtopics matter")


class ResearchBrief(BaseModel):
    """Output of the BriefWriter agent."""
    title: str = Field(description="Title of the research brief")
    summary: str = Field(description="Executive summary in 2-3 sentences")
    sections: list[dict[str, str]] = Field(
        description="List of sections, each with 'heading' and 'content' keys"
    )


# --- Agents ---

class TopicAnalyzer(BaseAgent[GlobalState, TopicAnalysis]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="topic_analyzer",
            system_prompt=(
                "You are a research analyst. Given a topic, identify exactly 3 key "
                "subtopics that would form the backbone of a comprehensive research brief. "
                "Choose subtopics that are distinct, substantive, and collectively cover "
                "the most important dimensions of the topic."
            ),
            output_type=TopicAnalysis,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> TopicAnalysis:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


class BriefWriter(BaseAgent[GlobalState, ResearchBrief]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="brief_writer",
            system_prompt=(
                "You are a technical writer who creates structured research briefs. "
                "Given a set of research questions, write a well-organized brief with "
                "a clear title, executive summary, and one section per question. "
                "Each section should have a descriptive heading and substantive content "
                "(at least 2-3 sentences per section). Use specific facts and evidence."
            ),
            output_type=ResearchBrief,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> ResearchBrief:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


print("Agents defined: TopicAnalyzer, BriefWriter")

Agents defined: TopicAnalyzer, BriefWriter


## 3. Build the pipeline

A `Pipeline` accepts a list of steps. Each step can be:
- A `BaseAgent` (auto-wrapped in `AgentStep`)
- An async callable (auto-wrapped in `FunctionStep`)
- A `(StepLike, StepConfig)` tuple for per-step error handling

Steps are chained sequentially: the output of step N becomes the input of step N+1. The pipeline handles coercion automatically via `_coerce_step()`.

In [3]:
from orqest.orchestration import Pipeline, StepConfig, ErrorStrategy


# Step 2 is a pure async function — no LLM needed.
# It receives a TopicAnalysis and returns a formatted string of research questions.
async def format_questions(analysis: TopicAnalysis) -> str:
    """Convert subtopics into numbered research questions."""
    questions = []
    for i, subtopic in enumerate(analysis.subtopics, 1):
        questions.append(f"{i}. What are the current developments, challenges, and "
                         f"future outlook for: {subtopic}?")

    header = f"Research Topic: {analysis.topic}\n\n"
    return header + "Research Questions:\n" + "\n".join(questions)


# Instantiate agents
analyzer = TopicAnalyzer(model=config.llm_model, api_key=config.llm_api_key)
writer = BriefWriter(model=config.llm_model, api_key=config.llm_api_key)

# Build the 3-step pipeline
pipeline = Pipeline(
    steps=[
        (analyzer, StepConfig(name="topic_analyzer", on_error=ErrorStrategy.STOP)),
        (format_questions, StepConfig(name="format_questions")),
        (writer, StepConfig(name="brief_writer", on_error=ErrorStrategy.STOP)),
    ],
    name="research_brief_pipeline",
)

print(f"Pipeline '{pipeline.name}' built with {len(pipeline._steps)} steps:")
for i, (step, cfg) in enumerate(pipeline._steps):
    print(f"  Step {i}: {cfg.name} (error strategy: {cfg.on_error.value})")

Pipeline 'research_brief_pipeline' built with 3 steps:
  Step 0: topic_analyzer (error strategy: stop)
  Step 1: format_questions (error strategy: stop)
  Step 2: brief_writer (error strategy: stop)


## 4. Run the pipeline

`pipeline.run()` executes all steps sequentially and returns the final output. The input string flows through:

1. **TopicAnalyzer** receives `"The impact of quantum computing on drug discovery"` and returns a `TopicAnalysis`
2. **format_questions()** receives the `TopicAnalysis` and returns a formatted string of research questions
3. **BriefWriter** receives the questions string and returns a `ResearchBrief`

In [4]:
import time

start = time.monotonic()
brief = await pipeline.run("The impact of quantum computing on drug discovery")
elapsed = time.monotonic() - start

print(f"Pipeline completed in {elapsed:.1f}s\n")
print(f"Title: {brief.title}\n")
print(f"Summary: {brief.summary}\n")
print("Sections:")
for section in brief.sections:
    print(f"\n  --- {section['heading']} ---")
    print(f"  {section['content'][:200]}...")

Pipeline completed in 10.5s

Title: The Impact of Quantum Computing on Drug Discovery

Summary: Quantum computing is poised to revolutionize drug discovery through advancements in quantum algorithms for molecular simulation and accelerated identification of drug candidates. Despite exciting progress and significant potential, several technical and practical challenges still limit widespread adoption in real-world applications. This brief explores the latest developments, ongoing challenges, and the future outlook across key areas related to quantum computing in drug discovery.

Sections:

  --- Quantum Algorithms for Molecular Simulation: Developments, Challenges, and Outlook ---
  Recent advances in quantum algorithms, such as Variational Quantum Eigensolver (VQE) and Quantum Phase Estimation (QPE), have enabled more accurate simulations of molecular interactions at the quantum...

  --- Acceleration of Drug Candidate Identification and Optimization: Developments, Challenges, and Outl

## 5. Pipeline with streaming

`pipeline.run_stream()` is an async iterator that yields `PipelineEvent` objects at each lifecycle point: pipeline start, step start, step complete, step skip, and pipeline complete. This gives you real-time observability into the pipeline's progress.

In [5]:
# Build a fresh pipeline (agents are stateless per-step, but let's be explicit)
stream_pipeline = Pipeline(
    steps=[
        (analyzer, StepConfig(name="topic_analyzer")),
        (format_questions, StepConfig(name="format_questions")),
        (writer, StepConfig(name="brief_writer")),
    ],
    name="research_brief_streamed",
)

print("Streaming pipeline events:\n")
async for event in stream_pipeline.run_stream("The role of CRISPR in treating genetic diseases"):
    ts = event.timestamp.strftime("%H:%M:%S")

    if event.event_type == "pipeline_start":
        print(f"[{ts}] PIPELINE START: {event.pipeline_name}")

    elif event.event_type == "step_start":
        print(f"[{ts}]   STEP {event.step_index} START: {event.step_name}")

    elif event.event_type == "step_complete":
        output = event.data.get("output")
        # Show a preview of the output
        if hasattr(output, "subtopics"):
            print(f"[{ts}]   STEP {event.step_index} COMPLETE: {event.step_name} "
                  f"-> {len(output.subtopics)} subtopics found")
        elif isinstance(output, str):
            print(f"[{ts}]   STEP {event.step_index} COMPLETE: {event.step_name} "
                  f"-> {len(output)} chars")
        elif hasattr(output, "title"):
            print(f"[{ts}]   STEP {event.step_index} COMPLETE: {event.step_name} "
                  f'-> brief "{output.title}"')

    elif event.event_type == "step_skip":
        print(f"[{ts}]   STEP {event.step_index} SKIPPED: {event.step_name}")

    elif event.event_type == "pipeline_complete":
        print(f"[{ts}] PIPELINE COMPLETE")

    elif event.event_type == "pipeline_error":
        print(f"[{ts}] PIPELINE ERROR at {event.step_name}: {event.error}")

Streaming pipeline events:

[17:26:00] PIPELINE START: research_brief_streamed
[17:26:00]   STEP 0 START: topic_analyzer


[17:26:02]   STEP 0 COMPLETE: topic_analyzer -> 3 subtopics found
[17:26:02]   STEP 1 START: format_questions
[17:26:02]   STEP 1 COMPLETE: format_questions -> 449 chars
[17:26:02]   STEP 2 START: brief_writer


[17:26:09]   STEP 2 COMPLETE: brief_writer -> brief "The Role of CRISPR in Treating Genetic Diseases: Mechanisms, Applications, and Considerations"
[17:26:09] PIPELINE COMPLETE


## 6. Error handling — SKIP strategy

What happens when a step fails? The `StepConfig.on_error` field controls the behavior:
- **STOP** (default) — raise `PipelineStepError` and halt the pipeline
- **SKIP** — swallow the error, emit a `step_skip` event, and continue with the previous step's output
- **RETRY** — retry the step up to `max_retries` times before raising

Below we add a step that intentionally fails. With `ErrorStrategy.SKIP`, the pipeline continues past it.

In [6]:
async def failing_enrichment(analysis: TopicAnalysis) -> TopicAnalysis:
    """Simulate an external API call that fails."""
    raise ConnectionError("External enrichment service unavailable")


# Pipeline: analyzer -> failing step (SKIP) -> format_questions -> writer
error_pipeline = Pipeline(
    steps=[
        (analyzer, StepConfig(name="topic_analyzer")),
        (failing_enrichment, StepConfig(name="enrichment_api", on_error=ErrorStrategy.SKIP)),
        (format_questions, StepConfig(name="format_questions")),
        (writer, StepConfig(name="brief_writer")),
    ],
    name="pipeline_with_skip",
)

print("Running pipeline with a failing step (SKIP strategy):\n")
async for event in error_pipeline.run_stream("Advances in renewable energy storage"):
    if event.event_type == "step_start":
        print(f"  Step {event.step_index}: {event.step_name} ... ", end="")
    elif event.event_type == "step_complete":
        print("DONE")
    elif event.event_type == "step_skip":
        print("SKIPPED (error handled gracefully)")
    elif event.event_type == "pipeline_complete":
        result = event.data.get("output")
        print(f"\nPipeline completed despite the failure!")
        print(f"Final output type: {type(result).__name__}")
        if hasattr(result, "title"):
            print(f"Brief title: {result.title}")

Running pipeline with a failing step (SKIP strategy):

  Step 0: topic_analyzer ... 

DONE
  Step 1: enrichment_api ... SKIPPED (error handled gracefully)
  Step 2: format_questions ... DONE
  Step 3: brief_writer ... 

DONE

Pipeline completed despite the failure!
Final output type: ResearchBrief
Brief title: Advances in Renewable Energy Storage: Technologies, Solutions, and Integration


## 7. RefinementLoop

The `RefinementLoop` iterates a step with evaluation feedback until quality passes or limits are reached. It needs three things:

1. **step** — the agent or function to iterate (here, a DraftWriter agent)
2. **evaluator** — a function that checks the output and returns an `EvalResult(passed, feedback, score)`
3. **state_updater** — a function that incorporates feedback into the next input

Our quality evaluator checks:
- Word count is above 50
- No banned filler words ("very", "really", "just")
- The text contains at least one piece of evidence (a number or statistic)

In [7]:
import re
from orqest.orchestration import RefinementLoop, EvalResult


# --- DraftWriter agent ---

class DraftOutput(BaseModel):
    """Output of the DraftWriter agent."""
    paragraph: str = Field(description="A single polished paragraph on the given topic")


class DraftWriter(BaseAgent[GlobalState, DraftOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="draft_writer",
            system_prompt=(
                "You are a precise technical writer. Write a single paragraph on the "
                "given topic. Requirements:\n"
                "- At least 50 words\n"
                "- Include specific numbers, statistics, or dates as evidence\n"
                "- Never use filler words: 'very', 'really', 'just'\n"
                "- Be concise and substantive\n\n"
                "If you receive feedback about a previous draft, revise accordingly."
            ),
            output_type=DraftOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> DraftOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


# --- Evaluator function ---

BANNED_WORDS = {"very", "really", "just"}

def evaluate_quality(output: DraftOutput) -> EvalResult:
    """Check draft quality: word count, banned words, evidence."""
    text = output.paragraph
    words = text.split()
    issues = []
    score = 1.0

    # Check word count
    if len(words) < 50:
        issues.append(f"Too short: {len(words)} words (need at least 50)")
        score -= 0.4

    # Check for banned filler words
    found_banned = [w for w in BANNED_WORDS if re.search(rf"\b{w}\b", text, re.IGNORECASE)]
    if found_banned:
        issues.append(f"Contains banned filler words: {', '.join(found_banned)}")
        score -= 0.2 * len(found_banned)

    # Check for evidence (numbers, percentages, dates)
    has_evidence = bool(re.search(r"\d+", text))
    if not has_evidence:
        issues.append("No evidence found (include numbers, statistics, or dates)")
        score -= 0.3

    passed = len(issues) == 0
    feedback = "Quality passed!" if passed else "Issues found:\n- " + "\n- ".join(issues)
    return EvalResult(passed=passed, feedback=feedback, score=max(0.0, score))


# --- State updater ---

def update_with_feedback(current_input: str, output: DraftOutput, eval_result: EvalResult) -> str:
    """Incorporate evaluator feedback into the next prompt."""
    return (
        f"Previous draft:\n{output.paragraph}\n\n"
        f"Evaluator feedback:\n{eval_result.feedback}\n\n"
        f"Please revise the paragraph to address all issues. "
        f"Keep the same topic but improve quality."
    )


print("RefinementLoop components defined:")
print("  - DraftWriter agent")
print("  - evaluate_quality() evaluator")
print("  - update_with_feedback() state updater")

RefinementLoop components defined:
  - DraftWriter agent
  - evaluate_quality() evaluator
  - update_with_feedback() state updater


In [8]:
draft_writer = DraftWriter(model=config.llm_model, api_key=config.llm_api_key)

loop = RefinementLoop(
    step=draft_writer,
    evaluator=evaluate_quality,
    state_updater=update_with_feedback,
    max_iterations=5,
)

loop_result = await loop.run(
    "Write a paragraph about the economic impact of large language models on the software industry."
)

print(f"Exit reason:  {loop_result.exit_reason}")
print(f"Iterations:   {loop_result.iterations}")
print(f"\nFinal paragraph:\n{loop_result.output.paragraph}")
print(f"\nIteration history:")
for record in loop_result.history:
    status = "PASSED" if record.eval_result.passed else "FAILED"
    print(f"  Iteration {record.iteration}: {status} "
          f"(score: {record.eval_result.score:.2f}, {record.duration_ms:.0f}ms)")
    if not record.eval_result.passed:
        print(f"    Feedback: {record.eval_result.feedback}")

Exit reason:  passed
Iterations:   1

Final paragraph:
Large language models (LLMs), such as OpenAI's GPT series, have significantly transformed the software industry by accelerating code generation, automating documentation, and streamlining customer support operations. According to a 2023 GitHub Copilot report, developers using AI-powered coding assistants completed tasks 55% faster and reported a 27% increase in overall productivity. The global AI software market, which reached approximately $138 billion in 2022 as reported by Grand View Research, is projected to grow at a compound annual growth rate of over 23% from 2023 to 2030, in part due to the adoption of LLMs. These models reduce development costs, enable rapid prototyping, and improve software quality, thereby reshaping competitive dynamics and amplifying economic output across the sector.

Iteration history:
  Iteration 1: PASSED (score: 1.00, 2608ms)


## 8. Assessment

Summary of timing, iteration counts, and quality scores across all runs.

In [9]:
# --- Pipeline assessment ---
print("=" * 60)
print("PIPELINE ASSESSMENT")
print("=" * 60)
print(f"\nPipeline execution time: {elapsed:.1f}s")
print(f"Pipeline steps:          {len(pipeline._steps)}")
print(f"Final output type:       {type(brief).__name__}")
print(f"Brief sections:          {len(brief.sections)}")

# --- RefinementLoop assessment ---
print(f"\n{'=' * 60}")
print("REFINEMENT LOOP ASSESSMENT")
print("=" * 60)
print(f"\nExit reason:    {loop_result.exit_reason}")
print(f"Iterations:     {loop_result.iterations} / {loop._max_iterations}")
total_loop_ms = sum(r.duration_ms for r in loop_result.history)
print(f"Total time:     {total_loop_ms / 1000:.1f}s")

scores = [r.eval_result.score for r in loop_result.history if r.eval_result.score is not None]
if scores:
    print(f"Score history:  {' -> '.join(f'{s:.2f}' for s in scores)}")
    print(f"Score delta:    {scores[-1] - scores[0]:+.2f} (first to last)")

# Final quality check on the output
final_eval = evaluate_quality(loop_result.output)
print(f"\nFinal quality:  {'PASS' if final_eval.passed else 'FAIL'} (score: {final_eval.score:.2f})")
word_count = len(loop_result.output.paragraph.split())
print(f"Word count:     {word_count}")
print(f"Has evidence:   {bool(re.search(r'[0-9]+', loop_result.output.paragraph))}")

found = [w for w in BANNED_WORDS if re.search(rf"\b{w}\b", loop_result.output.paragraph, re.IGNORECASE)]
print(f"Banned words:   {'none' if not found else ', '.join(found)}")

PIPELINE ASSESSMENT

Pipeline execution time: 10.5s
Pipeline steps:          3
Final output type:       ResearchBrief
Brief sections:          3

REFINEMENT LOOP ASSESSMENT

Exit reason:    passed
Iterations:     1 / 5
Total time:     2.6s
Score history:  1.00
Score delta:    +0.00 (first to last)

Final quality:  PASS (score: 1.00)
Word count:     116
Has evidence:   True
Banned words:   none


## What's happening under the hood

### Pipeline
1. `Pipeline.__init__()` coerces each entry via `_coerce_step()`: `BaseAgent` becomes `AgentStep`, async functions become `FunctionStep`
2. `AgentStep.execute()` creates a fresh `GlobalState`, converts the input to a prompt string (JSON for Pydantic models, `str()` otherwise), and calls `agent.run()`
3. `pipeline.run()` internally calls `run_stream()` and collects events — each step's output feeds directly as the next step's input
4. `StepConfig.on_error` controls failure behavior: `STOP` raises `PipelineStepError`, `SKIP` swallows the error and passes the previous output forward, `RETRY` retries up to `max_retries` times

### RefinementLoop
1. The loop calls `step.execute(current_input)` to produce output
2. The evaluator function receives the output and returns an `EvalResult(passed, feedback, score)`
3. If `passed=True`, the loop exits with `exit_reason="passed"`
4. If not, `state_updater(current_input, output, eval_result)` produces the next input with feedback baked in
5. The loop continues until: evaluator passes, `max_iterations` reached, `timeout` expires, or scores converge (if `convergence_window` is set)
6. `LoopResult` contains the final output, iteration count, exit reason, and full history of `IterationRecord` objects for observability